# Stage 2.5 — WaveNet-style hierarchical context

Stage 2.3's MLP took a flat concatenation of 3 character embeddings. To extend context to 8 characters with the same architecture, the first layer's input would be `8 * embed_dim = 80` numbers, and the layer would have to immediately mix them all at once. That's wasteful — the relationship between `prev_1` and `prev_2` is *local*; you don't need to look at `prev_8` to figure it out.

**WaveNet's idea**: build the network in a tree. Each layer combines pairs of adjacent embeddings, halving the sequence length while doubling the channel depth. After `log_2(block_size)` layers you've integrated the full context.

## What you'll ship

1. **Bigger context**: `block_size = 8`
2. **Hierarchical model**: instead of one flat MLP, a stack of `Linear → BatchNorm → Tanh` blocks where each block consumes pairs of timesteps
3. **Container modules** that mirror PyTorch's `nn.Sequential` etc. — Karpathy builds these from scratch in the lecture
4. **Final test-set evaluation**: this is the first stage that touches the held-out 10% test set. ~1.99 NLL
5. **Loss curves + samples**

## Reference

[Karpathy — Building makemore Part 5: Building a WaveNet](https://www.youtube.com/watch?v=t3YJ5hKiMQ0)

## Common traps

1. **Tensor shapes get harder to track.** Karpathy uses `(B, T, C)` for batch × time × channels everywhere. Print shapes between every block while debugging.
2. **The pairwise merge** is `x.view(B, T // 2, 2 * C)` — interleave consecutive timesteps along the channel dim. Mental model: each step in time `T/2` sees TWO timesteps from `T`.
3. **BatchNorm on `(B, T, C)`** needs to be careful — you want to normalize across batch + time, not per-(batch, time) cell. Karpathy fixes this manually in the lecture.
4. **Test set is sacred.** Use dev for tuning. Only run the test number once, at the very end.

## Plan

1. Define modular `Linear`, `BatchNorm1d`, `Tanh`, `Embedding`, `Flatten`, `FlattenConsecutive`, and `Sequential` containers — each with `parameters()` method, like PyTorch.
2. Build the WaveNet model: `Embedding → [Linear, BatchNorm1d, Tanh, FlattenConsecutive(2)] × log2(block_size) → Linear`
3. Train. Hit dev loss ≈ 1.99.
4. Run on test set, report final number.
5. Sample 20 names — they should look substantially more name-shaped than Stage 2.3's.

In [ ]:
import torch
import torch.nn.functional as F
from pathlib import Path

# Same vocab setup
words = Path('data/names.txt').read_text().splitlines()
chars = sorted(set(''.join(words)))
stoi = {c: i + 1 for i, c in enumerate(chars)}
stoi['.'] = 0
itos = {i: c for c, i in stoi.items()}
V = len(stoi)

# TODO: build_dataset with block_size=8, define modular containers, build WaveNet, train, test, sample.

## End-of-stage check

- [ ] Test set NLL ≈ 1.99 (and you only ran on the test set once)
- [ ] You can explain why hierarchical context is more parameter-efficient than a flat concat of 8 embeddings
- [ ] 20 samples look qualitatively name-like — most should be recognizable patterns
- [ ] One line to `NOTES.md` summarizing what changed in your mental model from Stage 2.1 to 2.5

**Block A finished.** Next: `03_nanogpt/` — same task, attention instead of convolution-style context.